### Connexion à la DB DuckDB

In [1]:
%pip install seaborn

Note: you may need to restart the kernel to use updated packages.


In [2]:
import duckdb
import os
from pathlib import Path
from typing import List
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import numpy as np
from tqdm import tqdm
import sklearn
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

### Connexion à la DB / Import des Data


In [3]:
# Store database at project root
DB_NAME = Path("/home/c-enjalbert/Documents/Github/MSPR/bloc_2/amazing/amazing.duckdb") 
# Go up one level from current directory to get to project root
data_folder = Path("..") / "data"
# For absolute certainty, you could use the absolute path
# data_folder = Path("/home/c-enjalbert/Documents/EPSI/MSPR/bloc_2/amazing/data")
con = duckdb.connect(str(DB_NAME))




















In [4]:
# 2. Query to list all tables in the database
# DuckDB specific way to list tables
tables_info = con.sql("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'main'
    ORDER BY table_name
""").df()

print(f"Found {len(tables_info)} tables in the database:\n")

if len(tables_info) > 0:
    for i, table_name in enumerate(tables_info['table_name']):
        print(f"{i+1}. {table_name}")
else:
    print("No tables found in the database.")

Found 4 tables in the database:

1. all_events
2. loaded_files
3. user_events
4. user_segments_kmeans


In [5]:
# 5. Alternative way to show all tables
print("List of all tables using DuckDB's connections.tables():")
con.sql("SHOW TABLES").show()

List of all tables using DuckDB's connections.tables():
┌──────────────────────┐
│         name         │
│       varchar        │
├──────────────────────┤
│ all_events           │
│ loaded_files         │
│ user_events          │
│ user_segments_kmeans │
└──────────────────────┘



In [6]:
# Examine the all_events table
print("First 10 rows of all_events table:")
all_events_data = con.sql("""
    SELECT * FROM all_events LIMIT 10
""")
all_events_data.show()


# Examine the loaded_files table
print("\nContents of loaded_files table:")
loaded_files_data = con.sql("""
    SELECT * FROM loaded_files
""")
loaded_files_data.show()


First 10 rows of all_events table:
┌─────────────────────┬────────────┬────────────┬─────────────────────┬────────────────────────────────┬─────────┬─────────┬───────────┬──────────────────────────────────────┐
│     event_time      │ event_type │ product_id │     category_id     │         category_code          │  brand  │  price  │  user_id  │             user_session             │
│      timestamp      │  varchar   │  varchar   │       varchar       │            varchar             │ varchar │ double  │  varchar  │               varchar                │
├─────────────────────┼────────────┼────────────┼─────────────────────┼────────────────────────────────┼─────────┼─────────┼───────────┼──────────────────────────────────────┤
│ 2019-12-01 00:00:00 │ view       │ 1005105    │ 2232732093077520756 │ construction.tools.light       │ apple   │ 1302.48 │ 556695836 │ ca5eefc5-11f9-450c-91ed-380285a0bc80 │
│ 2019-12-01 00:00:00 │ view       │ 22700068   │ 2232732091643068746 │ NULL         

In [7]:
# Examine the all_events table
print("First 10 rows of all_events table:")
all_events_data = con.sql("""
    SELECT * FROM all_events LIMIT 10
""")
all_events_data.show()

# Show count of records in all_events
record_count = con.sql("""
    SELECT COUNT(*) as total_events FROM all_events
""")
record_count.show()

# Examine the loaded_files table
print("\nContents of loaded_files table:")
loaded_files_data = con.sql("""
    SELECT * FROM loaded_files
""")
loaded_files_data.show()


First 10 rows of all_events table:
┌─────────────────────┬────────────┬────────────┬─────────────────────┬────────────────────────────────┬─────────┬─────────┬───────────┬──────────────────────────────────────┐
│     event_time      │ event_type │ product_id │     category_id     │         category_code          │  brand  │  price  │  user_id  │             user_session             │
│      timestamp      │  varchar   │  varchar   │       varchar       │            varchar             │ varchar │ double  │  varchar  │               varchar                │
├─────────────────────┼────────────┼────────────┼─────────────────────┼────────────────────────────────┼─────────┼─────────┼───────────┼──────────────────────────────────────┤
│ 2019-12-01 00:00:00 │ view       │ 1005105    │ 2232732093077520756 │ construction.tools.light       │ apple   │ 1302.48 │ 556695836 │ ca5eefc5-11f9-450c-91ed-380285a0bc80 │
│ 2019-12-01 00:00:00 │ view       │ 22700068   │ 2232732091643068746 │ NULL         

In [8]:
DB_NAME = "amazing.duckdb"
TABLE_EVENTS = "all_events"
TABLE_USER_EVENTS = "user_events"
SAMPLE_USER_PERCENT = 1
BATCH_SIZE = 1000 

### Import de la table DuckDB

In [9]:
# Chargement de users avec au moins 10 événements 
print("Chargement d'un échantillon d'utilisateurs actifs...")

# Afficher les tables disponibles dans la base de données
tables_df = con.execute("SELECT table_name FROM information_schema.tables WHERE table_schema = 'main'").fetch_df()
print("Tables disponibles dans la base de données :")
print(tables_df)

user_ids_df = con.execute(f"""
    SELECT user_id
    FROM all_events
    WHERE user_id IS NOT NULL
      AND event_time >= '2019-11-01 00:00:00'
      AND event_time < '2019-12-01 00:00:00'
    GROUP BY user_id
    HAVING COUNT(*) >= 10
""").fetch_df()

Chargement d'un échantillon d'utilisateurs actifs...
Tables disponibles dans la base de données :
             table_name
0            all_events
1          loaded_files
2           user_events
3  user_segments_kmeans


### Normalisation et Standardisation des données

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

sampled_user_ids = user_ids_df.sample(frac=SAMPLE_USER_PERCENT, random_state=42)['user_id'].tolist()

print(f"Nombre d'utilisateurs actifs échantillonnés : {len(sampled_user_ids)}")

#  Création des features utilisateurs batch par batch 
print("Création des features utilisateurs par batch...")

user_features_list = []

for i in tqdm(range(0, len(sampled_user_ids), BATCH_SIZE), desc="Avancement user features", ncols=100):
    batch_ids = sampled_user_ids[i:i+BATCH_SIZE]
    batch_ids_str = ",".join(f"'{uid}'" for uid in batch_ids)

    batch_query = f"""
    WITH
        base_events AS (
            SELECT
                user_id,
                event_type,
                event_time,
                price,
                category_code,
                LEAD(event_time) OVER (PARTITION BY user_id ORDER BY event_time) AS next_event_time
            FROM {TABLE_EVENTS}
            WHERE user_id IN ({batch_ids_str})       
            AND event_time >= '2019-11-01 00:00:00'
            AND event_time < '2019-12-01 00:00:00'
        ),
        features AS (
            SELECT
                user_id,
                COUNT(*) AS total_events,
                SUM(CASE WHEN event_type = 'view' THEN 1 ELSE 0 END) AS total_views,
                SUM(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) AS total_purchases,
                AVG(EXTRACT(EPOCH FROM (next_event_time - event_time))) AS avg_time_between_events,
                SUM(CASE WHEN event_type = 'purchase' THEN price ELSE 0 END) AS total_spent,
                COALESCE(AVG(CASE WHEN event_type = 'purchase' THEN price ELSE NULL END), 0) AS avg_basket,
                MAX(event_time) AS last_event_time
            FROM base_events
            GROUP BY user_id
    )
    SELECT
        *,
        CASE WHEN total_views > 0 THEN total_purchases * 1.0 / total_views ELSE 0 END AS conversion_rate,
        CASE WHEN (total_views + total_purchases) > 0 THEN total_purchases * 1.0 / (total_views + total_purchases) ELSE 0 END AS purchase_ratio,
        DATE_PART('day', CAST('2020-03-31 22:00:00' AS TIMESTAMP) - last_event_time) AS days_since_last_event
    FROM features
    """

    batch_features = con.execute(batch_query).fetch_df()

    # Filtre les User_id avec au moins 10 événements dans la batch 
    # Les Users avec moins de 10 événements resteront dans la table all_event avant d'être ajoutés, dans les batchs suivants, si le total d'évènement pour cette user_id dépasse le seuil.
    # Récupérer les user_id de cette batch
    valid_user_ids = con.execute(f"""
        SELECT user_id
        FROM {TABLE_EVENTS}
        WHERE user_id IN ({batch_ids_str})
        GROUP BY user_id
        HAVING COUNT(*) >= 10
    """).fetch_df()
    valid_user_ids = set(valid_user_ids["user_id"].astype(str))

    # Filtrage strict des user_id valides
    batch_features = batch_features[batch_features["user_id"].astype(str).isin(valid_user_ids)]

    user_features_list.append(batch_features)

# Fusionner tous les batchs
user_features = pd.concat(user_features_list, ignore_index=True)

# Vérification des NaN
print("Vérification des NaN")
nan_summary = user_features.isna().sum()
print("Résumé des NaN par colonne :")
print(nan_summary[nan_summary > 0])

users_with_nan = user_features[user_features.isna().any(axis=1)]
print(f"Nombre d'utilisateurs avec des NaN : {len(users_with_nan)}")
print("Exemples d'utilisateurs avec NaN :")
print(users_with_nan.head(10))

Nombre d'utilisateurs actifs échantillonnés : 1359785
Création des features utilisateurs par batch...


Avancement user features: 100%|█████████████████████████████████| 1360/1360 [30:48<00:00,  1.36s/it]


Vérification des NaN
Résumé des NaN par colonne :
Series([], dtype: int64)
Nombre d'utilisateurs avec des NaN : 0
Exemples d'utilisateurs avec NaN :
Empty DataFrame
Columns: [user_id, total_events, total_views, total_purchases, avg_time_between_events, total_spent, avg_basket, last_event_time, conversion_rate, purchase_ratio, days_since_last_event]
Index: []


In [11]:
# Standardisation
print("Standardisation des features...")
scaler = StandardScaler()
X_scaled = scaler.fit_transform(user_features.drop(columns=["last_event_time"]))

Standardisation des features...


In [12]:
user_features

,user_id,total_events,total_views,total_purchases,avg_time_between_events,total_spent,avg_basket,last_event_time,conversion_rate,purchase_ratio,days_since_last_event
0,553150326,22,20.0,2.0,124.857143,2797.50,1398.75,2019-11-01 14:52:00,0.100,0.090909,151
1,578219283,16,16.0,0.0,73.733333,0.00,0.00,2019-11-28 13:11:38,0.000,0.000000,124
2,567592718,14,13.0,0.0,170643.384615,0.00,0.00,2019-11-30 12:12:15,0.000,0.000000,122
3,558040401,27,27.0,0.0,78091.384615,0.00,0.00,2019-11-28 05:04:08,0.000,0.000000,124
4,512896421,11,11.0,0.0,184719.100000,0.00,0.00,2019-11-29 17:03:34,0.000,0.000000,123
...,...,...,...,...,...,...,...,...,...,...,...
1359780,570590014,23,23.0,0.0,69127.590909,0.00,0.00,2019-11-30 04:19:51,0.000,0.000000,122
1359781,573402271,24,24.0,0.0,495.130435,0.00,0.00,2019-11-19 13:03:17,0.000,0.000000,133
1359782,513593889,80,70.0,0.0,2168.708861,0.00,0.00,2019-11-15 01:15:30,0.000,0.000000,137
1359783,518727176,11,8.0,1.0,56.000000,28.29,28.29,2019-11-16 10:50:02,0.125,0.111111,136


In [ ]:
SELECTED_MONTHS ="11_2019"

In [14]:
#  Sauvegarde des résultats dans DuckDB
print(f"Sauvegarde dans {TABLE_USER_EVENTS}_{SELECTED_MONTHS}...")
con.execute(f"DROP TABLE IF EXISTS {TABLE_USER_EVENTS}_{SELECTED_MONTHS}")
con.register("temp_user_features", user_features)
con.execute(f"CREATE TABLE {TABLE_USER_EVENTS}_{SELECTED_MONTHS} AS SELECT * FROM temp_user_features")

Sauvegarde dans user_events_11_2019...


In [15]:
con.close()